## STAR SCHEMA  Business Logic and Queries

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS insurance_cat.gold

In [0]:
from pyspark.sql.functions import (
    col, current_date, current_timestamp,
    row_number, monotonically_increasing_id
)
from pyspark.sql.window import Window

df_dim_customers = spark.sql("""
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY customer_id) AS INT) AS customer_key,
        customer_id,
        first_name,
        last_name,
        gender,
        initcap(city) AS city,
        postcode,
        is_active,
        CASE
            WHEN year(current_date()) - year(CAST(date_of_birth AS DATE)) <= 24 THEN 'Under 25'
            WHEN year(current_date()) - year(CAST(date_of_birth AS DATE)) <= 34 THEN '25-34'
            WHEN year(current_date()) - year(CAST(date_of_birth AS DATE)) <= 44 THEN '35-44'
            WHEN year(current_date()) - year(CAST(date_of_birth AS DATE)) <= 54 THEN '45-54'
            ELSE '55+'
        END AS age_band
    FROM insurance_cat.silver.customers
    WHERE is_current = true       
""")
df_dim_customers \
    .withColumn("gold_ingestion_date", current_date()) \
    .withColumn("gold_ingestion_timestamp", current_timestamp()) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("insurance_cat.gold.dim_customers")

In [0]:
# ─────────────────────────────────────────────
# DIM POLICIES
# ─────────────────────────────────────────────

df_dim_policies = spark.sql("""
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY p.policy_id) AS INT) AS policy_key,
        p.policy_id,
        p.insurance_type,
        p.policy_subtype,
        p.payment_frequency,
        p.status,
        c.deductible_amount,
        c.max_coverage_limit,
        c.covers_accidental_damage,
        c.covers_theft,
        c.covers_fire,
        c.covers_flood,
        c.covers_liability,
        c.no_claims_discount
    FROM insurance_cat.silver.policies p
    LEFT JOIN insurance_cat.silver.coverages c
        ON p.policy_id = c.policy_id
    WHERE p.is_current = true
""")

df_dim_policies \
    .withColumn("gold_ingestion_date", current_date()) \
    .withColumn("gold_ingestion_timestamp", current_timestamp()) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("insurance_cat.gold.dim_policies")


In [0]:

# DIM AGENTS
# ─────────────────────────────────────────────

df_dim_agents = spark.sql("""
    SELECT
        CAST(ROW_NUMBER() OVER (ORDER BY agent_id) AS INT) AS agent_key,
        agent_id,
        first_name,
        last_name,
        region,
        specialisation,
        commission_rate,
        is_active
    FROM insurance_cat.silver.agents
    WHERE is_current = true
""")

df_dim_agents \
    .withColumn("gold_ingestion_date", current_date()) \
    .withColumn("gold_ingestion_timestamp", current_timestamp()) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("insurance_cat.gold.dim_agents")

In [0]:
# ─────────────────────────────────────────────
# DIM DATE
# ─────────────────────────────────────────────

df_dim_date = spark.sql("""
    SELECT DISTINCT
        CAST(DATE_FORMAT(claim_date, 'yyyyMMdd') AS INT) AS date_key,
        claim_date                                        AS full_date,
        YEAR(claim_date)                                  AS year,
        MONTH(claim_date)                                 AS month,
        DAY(claim_date)                                   AS day,
        QUARTER(claim_date)                               AS quarter,
        WEEKOFYEAR(claim_date)                            AS week_of_year,
        DATE_FORMAT(claim_date, 'MMMM')                   AS month_name,
        DATE_FORMAT(claim_date, 'EEEE')                   AS day_name,
        CASE
            WHEN DATE_FORMAT(claim_date, 'EEEE')
                IN ('Saturday','Sunday') THEN false
            ELSE true
        END                                               AS is_weekday
    FROM insurance_cat.silver.claims
    WHERE claim_date IS NOT NULL

    UNION

    SELECT DISTINCT
        CAST(DATE_FORMAT(start_date, 'yyyyMMdd') AS INT),
        start_date,
        YEAR(start_date),
        MONTH(start_date),
        DAY(start_date),
        QUARTER(start_date),
        WEEKOFYEAR(start_date),
        DATE_FORMAT(start_date, 'MMMM'),
        DATE_FORMAT(start_date, 'EEEE'),
        CASE
            WHEN DATE_FORMAT(start_date, 'EEEE')
                IN ('Saturday','Sunday') THEN false
            ELSE true
        END
    FROM insurance_cat.silver.policies
    WHERE start_date IS NOT NULL
""")

df_dim_date \
    .withColumn("gold_ingestion_date", current_date()) \
    .withColumn("gold_ingestion_timestamp", current_timestamp()) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("insurance_cat.gold.dim_date")

In [0]:

# ─────────────────────────────────────────────
# FACT CLAIMS
# ─────────────────────────────────────────────

df_fact_claims = spark.sql("""
    SELECT
        c.claim_id,
        CAST(DATE_FORMAT(c.claim_date, 'yyyyMMdd') AS INT) AS date_key,
        c.policy_id,
        c.customer_id,
        p.agent_id,

        -- Measures
        c.claim_amount,
        c.approved_amount,
        p.premium_amount,
        p.coverage_amount,
        c.claim_type,
        c.status                                    AS claim_status,
        p.insurance_type,

        -- Derived measures
        CASE
            WHEN c.status IN ('Approved','Settled') THEN 1
            ELSE 0
        END                                         AS is_approved,

        CASE
            WHEN c.approved_amount IS NOT NULL
            THEN ROUND(c.approved_amount / NULLIF(c.claim_amount, 0) * 100, 2)
            ELSE 0
        END                                         AS approval_rate_pct,

        DATEDIFF(c.settlement_date, c.claim_date)   AS days_to_settlement

    FROM insurance_cat.silver.claims c
    LEFT JOIN insurance_cat.silver.policies p
        ON c.policy_id = p.policy_id
    WHERE p.is_current = true
""")

df_fact_claims \
    .withColumn("gold_ingestion_date", current_date()) \
    .withColumn("gold_ingestion_timestamp", current_timestamp()) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("insurance_cat.gold.fact_claims")
